In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Dict, Any
from itertools import product
from collections import Counter
from tqdm import tqdm
from scipy.stats import chi2_contingency
from statsmodels.sandbox.stats.runs import runstest_1samp
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from pprint import pprint
from scipy.interpolate import interp1d
from scipy.stats import  gaussian_kde
from scipy.stats import ks_2samp

In [2]:
apple = pd.read_csv(r"D:\data\notebooks\week-10\cleaned_apple_high_low.csv")
apple['DATE'] = pd.to_datetime(apple['DATE'], errors='coerce')
apple = apple.sort_values('DATE')
apple.head()

,DATE,weekday,OPEN,CLOSE,VOL,HIGH,LOW
0,1984-09-07,Friday,0.10122,0.10122,97236149.0,0.10246,0.10000
1,1984-09-10,Monday,0.10122,0.10062,75471114.0,0.10153,0.09878
2,1984-09-11,Tuesday,0.10153,0.10246,177965367.0,0.10428,0.10153
3,1984-09-12,Wednesday,0.10246,0.09938,155467926.0,0.10306,0.09938
4,1984-09-13,Thursday,0.10490,0.10490,242135546.0,0.10520,0.10490


In [ ]:
def full_strategy_pipeline(params: Dict[str, Any]) -> Dict[str, Any]:
    df = params["df"]

    VALID_WEEKS       = params["VALID_WEEKS"]
    depth_grid        = params["depth_grid"]
    leaf_grid         = params["leaf_grid"]
    thresholds_tested = params["thresholds_tested"]
    FIXED             = params["FIXED"]

    alpha_p = params["alpha_p"]
    alpha_c = params["alpha_c"]
    p_min   = params["p_min"]
    c_min   = params["c_min"]

    n_trajectories = params["n_trajectories"]
    n_weeks        = params["n_weeks"]
    initial_bank   = params["initial_bank"]
    upper_thresh   = params["upper_thresh"]
    lower_thresh   = params["lower_thresh"]
    rng_seed       = params["rng_seed"]

    uniformity_binsize = params["uniformity_binsize"]

    cutoff_date       = params.get("cutoff_date", None)
    subset_start_date = params.get("subset_start_date", None)
    num_subsets       = params.get("num_subsets", 3)

    # --------------------------------------------------------
    # SR params + 9-feature grid (defaults if not provided)
    # --------------------------------------------------------
    ENVELOPE_A   = params.get("ENVELOPE_A", 1.0)
    ENVELOPE_B   = params.get("ENVELOPE_B", 100.0)
    SR_MAX_ITER  = params.get("SR_MAX_ITER", 60)
    SR_TOL       = params.get("SR_TOL", 1e-9)

    SR_SMOOTH_VALS = params.get("SR_SMOOTH_VALS", [2, 4, 12])
    SR_WINDOW_VALS = params.get("SR_WINDOW_VALS", [4, 26, 52])

    rng = np.random.default_rng(rng_seed)

    # --------------------------------------------------------
    # Cleaning and Weekly Dataset (BASE LOGIC; NO NORMALIZATION)
    # --------------------------------------------------------
    df = df.sort_values("DATE").reset_index(drop=True)
    df["DATE"] = pd.to_datetime(df["DATE"])

    if cutoff_date is not None:
        cutoff_dt = pd.to_datetime(cutoff_date)
        df = df[df["DATE"] >= cutoff_dt].reset_index(drop=True)

    df["weekday"] = df["DATE"].dt.weekday
    df["week"]    = df["DATE"].dt.to_period("W-SUN")

    tue_open = (
        df[df["weekday"] == 1]
        .groupby("week")["OPEN"]
        .first()
        .rename("tue_open")
    )
    thu_open = (
        df[df["weekday"] == 3]
        .groupby("week")["OPEN"]
        .first()
        .rename("thu_open")
    )

    weekly = pd.concat([tue_open, thu_open], axis=1)
    weekly["thu/tue"] = weekly["thu_open"] / weekly["tue_open"]
    weekly["week_type"] = (weekly["thu/tue"] > 1.0).astype(int)

    # --------------------------------------------------------
    # FULL SUPPORT/RESISTANCE ENVELOPE LOGIC (INLINED)
    # Builds 9 SR features: (TueOpen - Support)/(Resistance - Support)
    # for smoothing in {2,4,12} and window in {4,26,52}
    # --------------------------------------------------------
    df_sr = df.copy()
    df_sr["mid"] = (df_sr["HIGH"] + df_sr["LOW"]) / 2.0
    mon_raw = df_sr[df_sr["weekday"] == 0].copy().reset_index(drop=True)  # Mondays only

    def fit_line_weighted(t: np.ndarray, y: np.ndarray, g: np.ndarray, a_: float, b_: float) -> tuple[float, float]:
        X = np.column_stack([t, np.ones_like(t)])
        m, c = np.linalg.lstsq(X, y, rcond=None)[0]

        for _ in range(SR_MAX_ITER):
            r = y - (m * t + c)
            k = np.where(r > 0, a_, np.where(r < 0, b_, 0.0))
            w = g * k

            S_tt = np.sum(w * t * t)
            S_t  = np.sum(w * t)
            S_1  = np.sum(w)
            R_t  = np.sum(w * t * y)
            R_1  = np.sum(w * y)

            A = np.array([[S_tt, S_t],
                          [S_t,  S_1]])
            B = np.array([R_t, R_1])

            if abs(np.linalg.det(A)) < 1e-12:
                break

            m_new, c_new = np.linalg.solve(A, B)
            if abs(m_new - m) + abs(c_new - c) < SR_TOL:
                m, c = m_new, c_new
                break

            m, c = m_new, c_new

        return float(m), float(c)

    def build_envelope(mon_df: pd.DataFrame, smooth_window: int, window_weeks: int, a_: float, b_: float) -> pd.Series:
        # Smooth Monday mids
        m = mon_df.copy()
        m["mid_smooth"] = m["mid"].rolling(smooth_window, min_periods=smooth_window).mean()
        m = m.dropna(subset=["mid_smooth"]).reset_index(drop=True)

        # Global Monday index t
        m["t"] = np.arange(len(m), dtype=float)
        t_all = m["t"].to_numpy()
        y_all = m["mid_smooth"].to_numpy()

        env = np.full(len(m), np.nan)

        for i in range(len(m)):
            T = t_all[i]
            start = max(0, i - window_weeks)
            idx = slice(start, i + 1)

            t_win = t_all[idx]
            y_win = y_all[idx]
            g = (t_win - (T - window_weeks)) / window_weeks

            m_fit, c_fit = fit_line_weighted(t_win, y_win, g, a_, b_)
            env[i] = m_fit * T + c_fit

        m["envelope"] = env
        m["week"] = m["DATE"].dt.to_period("W-SUN")
        return m.set_index("week")["envelope"]

    SR_FEATURES = []
    for sw in SR_SMOOTH_VALS:
        for ww in SR_WINDOW_VALS:
            col = f"SR_{sw}_{ww}"
            SR_FEATURES.append(col)

            sup = build_envelope(mon_raw, smooth_window=sw, window_weeks=ww, a_=ENVELOPE_A, b_=ENVELOPE_B)
            res = build_envelope(mon_raw, smooth_window=sw, window_weeks=ww, a_=ENVELOPE_B, b_=ENVELOPE_A)

            # Align to weekly index (week periods)
            denom = (res - sup)
            pos = (weekly["tue_open"] - sup.reindex(weekly.index)) / denom.reindex(weekly.index)

            # invalid denominators -> NaN
            bad = denom.reindex(weekly.index) <= 0
            pos = pos.where(~bad, np.nan)

            weekly[col] = pos

    # Keep only rows where all 9 features exist + required weekly fields
    weekly_full = weekly.dropna(subset=SR_FEATURES + ["week_type", "thu/tue"])

    features = SR_FEATURES
    target   = "week_type"

    # --------------------------------------------------------
    # Rolling Train–Validate–Test (UNCHANGED)
    # --------------------------------------------------------
    def precision(tp, fp):
        return tp / (tp + fp) if (tp + fp) > 0 else 0.0

    def chattiness(tp, fp, fn):
        return (tp + fp) / (tp + fn) if (tp + fn) > 0 else 0.0

    def model_score(tp, fp, fn):
        P = precision(tp, fp)
        C = chattiness(tp, fp, fn)
        s = np.exp(alpha_p * (P - p_min) + alpha_c * (C - c_min))
        return 0.0 if np.isnan(s) or np.isinf(s) else float(s)

    from sklearn.tree import DecisionTreeClassifier
    from itertools import product
    from tqdm import tqdm

    TP = TN = FP = FN = 0
    weekly_best = []

    for t in tqdm(range(VALID_WEEKS + 1, len(weekly_full)), desc="Rolling simulation"):
        val_start  = max(0, t - VALID_WEEKS)
        training   = weekly_full.iloc[:val_start]
        validation = weekly_full.iloc[val_start:t]
        test       = weekly_full.iloc[[t]]

        if len(training[target].unique()) < 2:
            continue

        train_X, train_y = training[features], training[target]
        val_X, val_y     = validation[features], validation[target]
        test_X, test_y   = test[features], test[target]

        best_score  = -np.inf
        best_params = None
        best_model  = None

        for depth, leaf in product(depth_grid, leaf_grid):
            model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, **FIXED)
            model.fit(train_X, train_y)

            probs_val = model.predict_proba(val_X)[:, 1]
            for thr in thresholds_tested:
                preds_val = (probs_val > thr).astype(int)
                tp = ((preds_val == 1) & (val_y == 1)).sum()
                fp = ((preds_val == 1) & (val_y == 0)).sum()
                fn = ((preds_val == 0) & (val_y == 1)).sum()
                sc = model_score(tp, fp, fn)
                if sc > best_score:
                    best_score  = sc
                    best_params = (depth, leaf, thr)
                    best_model  = model

        if best_params is None:
            continue

        best_depth, best_leaf, best_thr = best_params

        p_hat = best_model.predict_proba(test_X)[0, 1]
        pred  = int(p_hat > best_thr)
        true  = int(test_y.iloc[0])

        if pred == 1 and true == 1:
            TP += 1; outcome = "TP"
        elif pred == 0 and true == 0:
            TN += 1; outcome = "TN"
        elif pred == 1 and true == 0:
            FP += 1; outcome = "FP"
        else:
            FN += 1; outcome = "FN"

        weekly_best.append({
            "Week": t,
            "True_Label": true,
            "Pred_Label": pred,
            "Outcome": outcome,
            "thu_tue": float(test["thu/tue"].iloc[0]),
        })

    df_final = pd.DataFrame(weekly_best)

    df_final["week_period"]     = weekly_full.index[df_final["Week"]]
    df_final["week_start_date"] = df_final["week_period"].dt.start_time

    # --------------------------------------------------------
    # Internal Metrics (UNCHANGED)
    # --------------------------------------------------------
    total = TP + TN + FP + FN
    prec_overall = precision(TP, FP)
    chat_overall = chattiness(TP, FP, FN)
    correctness_rate = (TP + TN) / total if total > 0 else 0.0
    pct_fp_positive = FP / (TP + FP) if (TP + FP) > 0 else 0.0

    df_final["correct"] = (df_final["True_Label"] == df_final["Pred_Label"]).astype(int)
    z_runs, p_runs = runstest_1samp(df_final["correct"], correction=False)

    randomness_test = {
        "H0": "Correctness is random in time.",
        "z": float(z_runs),
        "p": float(p_runs)
    }

    df_final["chunk"] = df_final.index // uniformity_binsize
    chi2, p_chi, dof, _ = chi2_contingency(pd.crosstab(df_final["chunk"], df_final["correct"]))

    uniformity_test = {
        "chi2": float(chi2),
        "p": float(p_chi),
        "dof": int(dof),
        "binsize": uniformity_binsize,
    }

    def longest_streak(seq, label):
        best = cur = 0
        for x in seq:
            if x == label:
                cur += 1
                best = max(best, cur)
            else:
                cur = 0
        return best

    longest_tp = longest_streak(df_final["Outcome"], "TP")
    longest_fp = longest_streak(df_final["Outcome"], "FP")

    TP_vals = df_final.loc[df_final["Outcome"] == "TP", "thu_tue"].values
    FP_vals = df_final.loc[df_final["Outcome"] == "FP", "thu_tue"].values
    tp_pct = (TP_vals - 1) * 100
    fp_pct = (FP_vals - 1) * 100
    mistake_asymmetry = float(tp_pct.mean() + fp_pct.mean()) if len(tp_pct) > 0 and len(fp_pct) > 0 else np.nan
    trade_frequency   = float((len(TP_vals) + len(FP_vals)) / len(df_final)) if len(df_final) > 0 else 0.0

    # --------------------------------------------------------
    # Subset Construction with num_subsets Logic (UNCHANGED)
    # --------------------------------------------------------
    if subset_start_date is not None:
        ss_dt = pd.to_datetime(subset_start_date)
        mask = df_final["week_start_date"] >= ss_dt
        if mask.any():
            base_start = mask.idxmax()
        else:
            base_start = len(df_final)
    else:
        base_start = 0

    last_start = len(df_final) - n_weeks
    if last_start < base_start:
        raise ValueError("Not enough data to produce even the final 100-week subset.")

    if num_subsets < 0:
        raise ValueError("num_subsets must be >= 0")

    if num_subsets > 0:
        offsets = np.linspace(0, last_start - base_start, num_subsets + 2, dtype=int)
        start_points = offsets[0:num_subsets] + base_start
    else:
        start_points = []

    subsets = []
    for s in start_points:
        subsets.append((s, df_final.iloc[s:s + n_weeks]))

    subsets.append((last_start, df_final.iloc[last_start:last_start + n_weeks]))

    # --------------------------------------------------------
    # Historical Monte Carlo (UNCHANGED)
    # --------------------------------------------------------
    outcomes_arr = np.array(["TP", "FP", "FN", "TN"])

    def build_sampler(vals):
        vals = np.sort(vals)
        if len(vals) == 0:
            return None, None
        cdf = np.arange(1, len(vals) + 1) / len(vals)
        return vals, cdf

    def sample(vals, cdf):
        u = rng.random()
        idx = np.searchsorted(cdf, u)
        return vals[min(idx, len(vals) - 1)]

    def run_actual(sub):
        bank = initial_bank
        for _, row in sub.iterrows():
            if row["Outcome"] in ("TP", "FP"):
                bank *= row["thu_tue"]
            if bank >= upper_thresh or bank <= lower_thresh:
                break
        return bank

    def run_mc_block(p, tp_vals, tp_cdf, fp_vals, fp_cdf):
        cdf = np.cumsum(p)
        final = np.empty(n_trajectories)
        for i in range(n_trajectories):
            bank = initial_bank
            for _ in range(n_weeks):
                r = rng.random()
                idx = np.searchsorted(cdf, r)
                outcome = outcomes_arr[idx]
                if outcome == "TP" and tp_vals is not None:
                    bank *= sample(tp_vals, tp_cdf)
                elif outcome == "FP" and fp_vals is not None:
                    bank *= sample(fp_vals, fp_cdf)
                if bank >= upper_thresh or bank <= lower_thresh:
                    break
            final[i] = bank
        return final

    all_sims = []
    actual_balances = []
    null_percentiles = []

    for i, (s, sub) in enumerate(subsets):
        if i == 0:
            continue

        history = df_final.iloc[:s]

        tp_hist = history.loc[history["Outcome"] == "TP", "thu_tue"].values
        fp_hist = history.loc[history["Outcome"] == "FP", "thu_tue"].values

        if len(tp_hist) < 2 or len(fp_hist) < 2:
            continue

        tp_vals_hist, tp_cdf_hist = build_sampler(tp_hist)
        fp_vals_hist, fp_cdf_hist = build_sampler(fp_hist)

        p_hist = history["Outcome"].value_counts(normalize=True).reindex(outcomes_arr, fill_value=0).values

        sims = run_mc_block(p_hist, tp_vals_hist, tp_cdf_hist, fp_vals_hist, fp_cdf_hist)
        all_sims.append(sims)

        actual = run_actual(sub)
        actual_balances.append(actual)

        null_percentiles.append(float(np.mean(sims <= actual)))

    if len(all_sims) > 0:
        sim_all = np.concatenate(all_sims)
    else:
        sim_all = np.array([initial_bank])

    actual_balances = np.array(actual_balances) if len(actual_balances) > 0 else np.array([initial_bank])
    null_percentiles = null_percentiles if len(null_percentiles) > 0 else [0.5]

    ks_d, ks_p = ks_2samp(actual_balances, sim_all)

    simulated_mean   = float(sim_all.mean())
    simulated_median = float(np.median(sim_all))
    avg_null         = float(np.mean(null_percentiles))

    # --------------------------------------------------------
    # Future Monte Carlo – unchanged (FULL DATASET df_final)
    # --------------------------------------------------------
    tp_all = df_final.loc[df_final["Outcome"] == "TP", "thu_tue"].values
    fp_all = df_final.loc[df_final["Outcome"] == "FP", "thu_tue"].values

    if len(tp_all) > 1:
        tp_vals_all, tp_cdf_all = build_sampler(tp_all)
    else:
        tp_vals_all, tp_cdf_all = (None, None)

    if len(fp_all) > 1:
        fp_vals_all, fp_cdf_all = build_sampler(fp_all)
    else:
        fp_vals_all, fp_cdf_all = (None, None)

    p_all = df_final["Outcome"].value_counts(normalize=True).reindex(outcomes_arr, fill_value=0).values

    fut = run_mc_block(p_all, tp_vals_all, tp_cdf_all, fp_vals_all, fp_cdf_all)

    future_mean     = float(fut.mean())
    future_median   = float(np.median(fut))
    prob_above_init = float(np.mean(fut > initial_bank))
    prob_success    = float(np.mean(fut >= upper_thresh))
    prob_failure    = float(np.mean(fut <= lower_thresh))
    prob_uncertain  = float(1 - prob_success - prob_failure)

    # --------------------------------------------------------
    # Baseline Comparison (all subsets) (UNCHANGED)
    # --------------------------------------------------------
    ratio_always   = []
    ratio_random   = []
    ratio_alt      = []
    ratio_weighted = []

    for s, sub in subsets:
        model_bal = run_actual(sub)

        b = initial_bank
        for r in sub["thu_tue"]:
            b *= r
            if b >= upper_thresh or b <= lower_thresh:
                break
        ratio_always.append(model_bal / b if b != 0 else np.nan)

        ch_prob = len(sub.loc[sub["Outcome"].isin(["TP", "FP"])]) / len(sub)
        b = initial_bank
        for r in sub["thu_tue"]:
            if rng.random() < ch_prob:
                b *= r
        ratio_random.append(model_bal / b if b != 0 else np.nan)

        b = initial_bank
        for i_idx, r in enumerate(sub["thu_tue"]):
            if i_idx % 2 == 0:
                b *= r
        ratio_alt.append(model_bal / b if b != 0 else np.nan)

        good_rate = float((sub["thu_tue"] > 1).mean())
        b = initial_bank
        for r in sub["thu_tue"]:
            if rng.random() < good_rate:
                b *= r
        ratio_weighted.append(model_bal / b if b != 0 else np.nan)

    # --------------------------------------------------------
    # Report Card (before rounding) (UNCHANGED KEYS)
    # --------------------------------------------------------
    report = {
        "historical_mc": {
            "simulated_mean": simulated_mean,
            "simulated_median": simulated_median,
            "ks_distance": float(ks_d),
            "ks_p_value": float(ks_p),
            "average_null_percentile": avg_null
        },
        "future_mc": {
            "future_mean": future_mean,
            "future_median": future_median,
            "prob_above_initial": prob_above_init,
            "prob_success": prob_success,
            "prob_failure": prob_failure,
            "prob_uncertain": prob_uncertain
        },
        "internal_metrics": {
            "precision_overall": prec_overall,
            "chattiness_overall": chat_overall,
            "correctness_rate": correctness_rate,
            "trade_frequency": trade_frequency,
            "mistake_asymmetry_%": mistake_asymmetry,
            "longest_TP_streak": longest_tp,
            "longest_FP_streak": longest_fp,
            "%FP_when_predicted_positive": pct_fp_positive
        },
        "baseline_comparison": {
            "vs_always_trade": float(np.nanmean(ratio_always)),
            "vs_random_trader": float(np.nanmean(ratio_random)),
            "vs_alternate_trader": float(np.nanmean(ratio_alt)),
            "vs_weighted_coin": float(np.nanmean(ratio_weighted))
        },
        "uniformity_test": uniformity_test,
        "randomness_test": randomness_test,
    }

    # --------------------------------------------------------
    # Round all floats in report to 2 decimal places (UNCHANGED)
    # --------------------------------------------------------
    def round_2(obj):
        if isinstance(obj, float):
            return round(obj, 2)
        if isinstance(obj, dict):
            return {k: round_2(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [round_2(x) for x in obj]
        return obj

    report = round_2(report)
    pprint(report)
    return report


In [9]:
params = {
    # =========================================================
    # DATA
    # =========================================================
    "df": apple,   # DataFrame with DATE, OPEN, HIGH, LOW columns

    # =========================================================
    # ROLLING MODEL / CV
    # =========================================================
    "VALID_WEEKS": 52,
    "depth_grid": [2, 3, 4, 6],
    "leaf_grid": [2, 3, 4, 6],
    "thresholds_tested": np.linspace(0.01, 0.99, 99),

    "FIXED": {
        "criterion": "entropy",
        "min_samples_split": 6,
        "class_weight": "balanced",
        "random_state": 42,
    },

    # =========================================================
    # SCORING FUNCTION (UNCHANGED)
    # =========================================================
    "alpha_p": 1.0,
    "alpha_c": 0.01,
    "p_min": 0.55,
    "c_min": 0.10,

    # =========================================================
    # MONTE CARLO
    # =========================================================
    "n_trajectories": 100000,
    "n_weeks": 100,
    "initial_bank": 100.0,
    "upper_thresh": 200.0,
    "lower_thresh": 60.0,
    "rng_seed": 42,

    # =========================================================
    # STATISTICAL TESTING
    # =========================================================
    "uniformity_binsize": 104,

    # =========================================================
    # DATA CUTS / SUBSETS
    # =========================================================
    "cutoff_date": "1990-01-01",
    "subset_start_date": "2000-01-01",
    "num_subsets": 5,

    # =========================================================
    # SUPPORT / RESISTANCE – CORE ENVELOPE LOGIC
    # =========================================================
    "ENVELOPE_A": 1.0,        # penalty for points ABOVE line (support)
    "ENVELOPE_B": 100.0,      # penalty for points BELOW line (support)
    "SR_MAX_ITER": 60,
    "SR_TOL": 1e-9,

    # =========================================================
    # SUPPORT / RESISTANCE – 9 FEATURE GRID
    # =========================================================
    # (These generate SR_2_4, SR_2_26, SR_2_52,
    #  SR_4_4, SR_4_26, SR_4_52,
    #  SR_12_4, SR_12_26, SR_12_52)
    "SR_SMOOTH_VALS": [2, 4, 12],
    "SR_WINDOW_VALS": [4, 26, 52],
}
report = full_strategy_pipeline(params)


Rolling simulation: 100%|██████████| 1577/1577 [31:03<00:00,  1.18s/it]


{'baseline_comparison': {'vs_alternate_trader': 1.14,
                         'vs_always_trade': 1.07,
                         'vs_random_trader': 1.22,
                         'vs_weighted_coin': 1.1},
 'future_mc': {'future_mean': 109.79,
               'future_median': 105.77,
               'prob_above_initial': 0.6,
               'prob_failure': 0.01,
               'prob_success': 0.01,
               'prob_uncertain': 0.98},
 'historical_mc': {'average_null_percentile': 0.66,
                   'ks_distance': 0.47,
                   'ks_p_value': 0.15,
                   'simulated_mean': 109.71,
                   'simulated_median': 104.5},
 'internal_metrics': {'%FP_when_predicted_positive': 0.48,
                      'chattiness_overall': 0.62,
                      'correctness_rate': 0.5,
                      'longest_FP_streak': 6,
                      'longest_TP_streak': 8,
                      'mistake_asymmetry_%': 0.32,
                      'precision_overa